# 语音处理端 · Google Colab 版（tts-with-rvc）

在 Colab 上运行本项目自带的服务端：**Edge TTS + RVC**。启动后用 Cloudflare 临时隧道
暴露到公网，AstrBot 插件把 `tts_server.url` 指过去就能用语音回复。

## 使用步骤（从上往下依次运行每个单元格）

1. **参数设置** —— 填 API Key、选择 RVC 模型来源
2. **安装依赖** —— 约 2~4 分钟（首次）
3. **写入服务端代码** —— 已内嵌，无需上传源码
4. **准备模型与权重** —— 你的 `.pth`（Google Drive / 上传 / 直链）+ hubert/rmvpe 自动下载
5. **生成配置**
6. **启动服务** —— 看到 `服务就绪` 即成功
7. **建立公网隧道** —— 拿到 `https://xxxx.trycloudflare.com`，填进 AstrBot 插件
8. **保持运行** —— 这个单元格一直跑着，会话就不会空转断开
9. **常用操作** —— 看日志 / 试听 / 停止

## 重要限制（务必先读）

| 项目 | 说明 |
| --- | --- |
| 会话时长 | Colab 免费版最长约 12 小时；超时或关掉浏览器后服务就没了 |
| 公网地址 | **每次运行都是新地址**（`*.trycloudflare.com`），插件里的 URL 要跟着改 |
| 显卡 | 免费版通常是 T4（16GB 显存），跑 RVC 完全够用；CPU 模式也能跑，只是慢 |
| 模型文件 | Colab 每次都是全新环境，**模型放 Google Drive** 最省事（本 notebook 会自动挂载） |
| 首次请求 | 自动下载 hubert(180MB) + rmvpe(173MB) 并加载模型，约 30~60 秒 |
| 安全 | 已开启 API Key 鉴权；地址 + Key 不要外传 |


In [ ]:
# ============================ 参数设置（改这里）============================
API_KEY = "colab-tts-key"     # AstrBot 插件里要填同一个值（建议改成随机字符串）
PORT = 8080

# RVC 模型来源，四选一：
#   "drive"  → 已把 .pth（和 .index）放到 Google Drive 的 ColabVoice 目录（推荐）
#   "upload" → 运行时手动上传（55MB 左右，稍慢）
#   "url"    → 从直链下载（填 MODEL_URL / INDEX_URL）
#   "none"   → 先不用 RVC，只跑 Edge TTS（验证链路用）
MODEL_SOURCE = "drive"

DRIVE_DIR = "/content/drive/MyDrive/ColabVoice"
MODEL_FILE = "Shaonian.pth"
INDEX_FILE = "Shaonian.index"      # 没有索引文件就留空字符串 ""
MODEL_URL = ""                     # MODEL_SOURCE="url" 时填 .pth 直链
INDEX_URL = ""                     # 可选

F0_METHOD = "rmvpe"                # rmvpe 质量最好；想更快可改 pm / dio
PITCH = 5                          # RVC 变调半音数
MAX_TEXT_LENGTH = 300              # 与插件侧 voice.max_text_length 保持一致
EXPIRE_MINUTES = 10                # 服务端音频保留时间（分钟）

WORKDIR = "/content/tts-server"    # 服务端运行目录（每次会话重新生成）
print(f"参数就绪：端口={PORT} 模型来源={MODEL_SOURCE} f0={F0_METHOD} pitch={PITCH}")


In [ ]:
import os, subprocess, sys, time


def run(cmd, check=True, quiet=False):
    """执行命令并回显输出（quiet=True 时只在失败时打印）。"""
    p = subprocess.Popen(cmd, shell=isinstance(cmd, str),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in p.stdout:
        lines.append(line)
        if not quiet:
            print(line, end="")
    rc = p.wait()
    if check and rc != 0:
        print("".join(lines[-40:]))
        raise RuntimeError(f"命令失败({rc}): {cmd}")
    return rc


print("=== 1/3 检查 GPU ===")
run([sys.executable, "-c",
     "import torch;print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(),"
     "'|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"])

print("\n=== 2/3 检查 ffmpeg（Colab 通常已自带）===")
if os.system("ffmpeg -version >/dev/null 2>&1") != 0:
    run("apt-get -qq update && apt-get -qq install -y ffmpeg", quiet=True)
print(subprocess.run("ffmpeg -version | head -1", shell=True,
                     capture_output=True, text=True).stdout.strip())

print("\n=== 3/3 安装服务端依赖（约 2~4 分钟）===")
run([sys.executable, "-m", "pip", "install", "-q",
     "fastapi", "uvicorn[standard]", "pydantic", "PyYAML", "httpx"])
rc = run([sys.executable, "-m", "pip", "install", "-q", "tts-with-rvc>=0.1.9"], check=False)
if rc != 0:
    # 少数环境里 praat-parselmouth 需要现场编译（本项目默认用 rmvpe，用不到它）
    print("\n[!] tts-with-rvc 安装失败，改为跳过 praat-parselmouth 再装一次…")
    run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "tts-with-rvc>=0.1.9"])
    run([sys.executable, "-m", "pip", "install", "-q",
         "huggingface_hub", "av", "nest_asyncio", "edge-tts", "numpy==1.26.0",
         "librosa==0.9.1", "faiss-cpu==1.10.0", "soundfile>=0.12.1", "ffmpeg-python>=0.2.0",
         "resampy>=0.4.2", "scikit-learn", "tqdm>=4.63.1", "audioread", "pyworld==0.3.5",
         "torchcrepe==0.0.20", "numba==0.60.0", "cffi<2.0.0", "einops", "local_attention",
         "fairseq-fixed"])

print("\n=== 自检 ===")
run([sys.executable, "-c",
     "import torch, fastapi, tts_with_rvc;"
     "print('torch', torch.__version__);print('CUDA 可用:', torch.cuda.is_available())"])


In [ ]:
import base64, gzip, io, os, tarfile

# 服务端代码已内嵌在本单元格里（main.py / config.py / engine.py / storage.py）
PAYLOAD_B64 = "..."

os.makedirs(f"{WORKDIR}/app", exist_ok=True)
raw = gzip.decompress(base64.b64decode(PAYLOAD_B64))
with tarfile.open(fileobj=io.BytesIO(raw), mode="r") as tar:
    tar.extractall(f"{WORKDIR}/app")

print("已写入:", ", ".join(sorted(os.listdir(f"{WORKDIR}/app"))))


In [ ]:
import os, shutil, subprocess, sys, urllib.request

os.makedirs(f"{WORKDIR}/models", exist_ok=True)
model_path = ""
index_path = ""

if MODEL_SOURCE == "drive":
    from google.colab import drive

    drive.mount("/content/drive")
    src_model = os.path.join(DRIVE_DIR, MODEL_FILE)
    if os.path.exists(src_model):
        shutil.copy(src_model, f"{WORKDIR}/models/{MODEL_FILE}")
        model_path = MODEL_FILE
        print("已从 Drive 复制模型:", MODEL_FILE)
        if INDEX_FILE and os.path.exists(os.path.join(DRIVE_DIR, INDEX_FILE)):
            shutil.copy(os.path.join(DRIVE_DIR, INDEX_FILE), f"{WORKDIR}/models/{INDEX_FILE}")
            index_path = INDEX_FILE
            print("已从 Drive 复制索引:", INDEX_FILE)
    else:
        print(f"[!] Drive 里没找到 {src_model}，本次只用 Edge TTS（不跑 RVC）")
elif MODEL_SOURCE == "upload":
    from google.colab import files

    print("请选择 .pth（可以同时选中 .index）：")
    uploaded = files.upload()
    for name, data in uploaded.items():
        with open(f"{WORKDIR}/models/{name}", "wb") as fh:
            fh.write(data)
        if name.lower().endswith(".pth"):
            model_path = name
        elif name.lower().endswith(".index"):
            index_path = name
    print("上传完成:", model_path, index_path)
elif MODEL_SOURCE == "url":
    if not MODEL_URL:
        raise ValueError('MODEL_SOURCE="url" 时必须填 MODEL_URL')
    urllib.request.urlretrieve(MODEL_URL, f"{WORKDIR}/models/{MODEL_FILE}")
    model_path = MODEL_FILE
    if INDEX_URL:
        urllib.request.urlretrieve(INDEX_URL, f"{WORKDIR}/models/{INDEX_FILE}")
        index_path = INDEX_FILE
    print("已从直链下载模型")
else:
    print("MODEL_SOURCE=none：只跑 Edge TTS（不加载 RVC 模型）")

# hubert / rmvpe 权重：tts-with-rvc 会在"当前工作目录"里找这两个文件
for asset in ("hubert_base.pt", "rmvpe.pt"):
    dst = os.path.join(WORKDIR, asset)
    if not os.path.exists(dst):
        print(f"下载 {asset} …")
        url = f"https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/{asset}"
        subprocess.run(["wget", "-q", "-O", dst, url], check=True)
    print(f"  {asset}: {os.path.getsize(dst) / 1024 / 1024:.1f} MB")

os.chdir(WORKDIR)
print("\n模型:", model_path or "(未配置)", "| 索引:", index_path or "(无)")


In [ ]:
import os

config_text = f"""# Colab 语音服务配置（本文件由 notebook 生成；手改后重启服务生效）
server:
  host: "0.0.0.0"
  port: {PORT}
  log_level: "INFO"

security:
  api_key: "{API_KEY}"

tts:
  source: "edgetts"
  speaker: "zh-CN-YunxiNeural"
  pitch: {PITCH}

rvc:
  enabled: {str(bool(model_path)).lower()}
  model: "{model_path}"
  model_dir: "./models"
  index: "{index_path}"
  f0_method: "{F0_METHOD}"
  device: "auto"
  min_vram_mb: 2500
  allow_cpu_fallback: true
  is_half: true
  preload: true

storage:
  output_dir: "./output"
  expire_minutes: {EXPIRE_MINUTES}
  cleanup_interval_minutes: 2
  cache_by_text: true
  max_text_length: 2000

queue:
  max_concurrent: 1
  max_queue_size: 16
  timeout: 180

audio:
  output_format: "wav"
"""

os.makedirs(WORKDIR, exist_ok=True)
with open(f"{WORKDIR}/config.yaml", "w", encoding="utf-8") as fh:
    fh.write(config_text)
print(config_text)


In [ ]:
import json, os, subprocess, sys, time, urllib.request

os.makedirs(f"{WORKDIR}/logs", exist_ok=True)
os.makedirs(f"{WORKDIR}/output", exist_ok=True)
log_file = open(f"{WORKDIR}/logs/server.log", "w", encoding="utf-8")
env = {**os.environ, "PYTHONUNBUFFERED": "1", "TMPDIR": "/tmp"}

server = subprocess.Popen(
    [sys.executable, "-s", f"{WORKDIR}/app/main.py", "--config", f"{WORKDIR}/config.yaml"],
    cwd=WORKDIR, stdout=log_file, stderr=subprocess.STDOUT, env=env)
print("服务进程 PID:", server.pid, "（首次要加载模型，稍等几十秒）")


def health():
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/health", timeout=5) as resp:
            return json.loads(resp.read())
    except Exception:
        return None


ready = False
for _ in range(150):
    if server.poll() is not None:
        break
    info = health()
    if info and info.get("success"):
        eng = info["engine"]
        name = os.path.basename(eng.get("model") or "") or "-"
        print(f"✅ 服务就绪：device={eng['device']} is_half={eng['is_half']} "
              f"model={name} f0={eng['f0_method']}")
        ready = True
        break
    time.sleep(2)

if not ready:
    print("❌ 启动失败，日志尾部：")
    with open(f"{WORKDIR}/logs/server.log", encoding="utf-8", errors="ignore") as fh:
        print("".join(fh.readlines()[-30:]))


In [ ]:
import os, re, subprocess, time

CLOUDFLARED = "/usr/local/bin/cloudflared"
if not os.path.exists(CLOUDFLARED):
    print("下载 cloudflared …")
    subprocess.run(
        "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/"
        f"cloudflared-linux-amd64 -O {CLOUDFLARED} && chmod +x {CLOUDFLARED}",
        shell=True, check=True)

print("启动隧道 …")
tunnel_log_path = "/content/cloudflared.log"
tunnel_log = open(tunnel_log_path, "w", encoding="utf-8")
tunnel = subprocess.Popen(
    [CLOUDFLARED, "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=tunnel_log, stderr=subprocess.STDOUT)

public_url = None
for _ in range(60):
    time.sleep(1)
    text = open(tunnel_log_path, encoding="utf-8", errors="ignore").read()
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
    if found:
        public_url = found.group(0)
        break

if public_url:
    print("\n" + "=" * 68)
    print("公网地址（复制下面这段填进 AstrBot 插件配置）")
    print("=" * 68)
    print(f"""
tts_server:
  url: "{public_url}"
  api_key: "{API_KEY}"
  delivery: "file"

voice:
  enabled: true
  max_text_length: {MAX_TEXT_LENGTH}
  timeout: 90
  max_concurrent: 2
""")
    print("自检命令：")
    print(f'  curl -H "Authorization: Bearer {API_KEY}" {public_url}/api/health')
    print("=" * 68)
else:
    print("❌ 没拿到公网地址，隧道日志尾部：")
    print(open(tunnel_log_path, encoding="utf-8", errors="ignore").read()[-1500:])


In [ ]:
# 让这个单元格一直跑着：既能看到实时状态，也能让 Colab 认为会话在用
import json, time, urllib.request

while True:
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/health", timeout=5) as resp:
            info = json.loads(resp.read())
        eng = info["engine"]
        stats = eng["stats"]
        print(f"[{time.strftime('%H:%M:%S')}] {info['status']} | {eng['device']} | "
              f"推理 {stats['total']} 次（成功 {stats['success']} / 失败 {stats['failed']}）| "
              f"最近 {stats.get('last_duration')}s", flush=True)
    except Exception as exc:
        print(f"[{time.strftime('%H:%M:%S')}] 服务不可用: {exc}", flush=True)
    time.sleep(60)


In [ ]:
# 查看服务端日志尾部（排错用）
with open(f"{WORKDIR}/logs/server.log", encoding="utf-8", errors="ignore") as fh:
    print("".join(fh.readlines()[-40:]))


In [ ]:
# 在 Colab 里直接试听一条（走完整 TTS + RVC 流程）
import json, urllib.request
from IPython.display import Audio, display

text = "你好，这是运行在 Google Colab 上的语音合成测试。"
req = urllib.request.Request(
    f"http://127.0.0.1:{PORT}/api/tts/file",
    data=json.dumps({"text": text}).encode(),
    headers={"Content-Type": "application/json", "Authorization": f"Bearer {API_KEY}"})
with urllib.request.urlopen(req, timeout=300) as resp:
    audio = resp.read()
with open("/content/test.wav", "wb") as fh:
    fh.write(audio)
print(f"已生成 /content/test.wav（{len(audio) / 1024:.1f} KB）")
display(Audio("/content/test.wav"))


In [ ]:
# 停止服务与隧道（想重新开始就从头再运行一遍）
import subprocess

for pattern in ("cloudflared", "app/main.py"):
    subprocess.run(f"pkill -f '{pattern}'", shell=True)
print("已停止")
